# Text generality

Reliability edits each sample on the original question. Text generality asks the same image again with a rephrased question and checks whether the human correction still comes back.

The numbered answer format is fixed. Only the question sentence changes. Each edit gets one sentence, sampled with a fixed seed. The codebook is built on the original question; retrieval and generation use the rephrased one.

`dermedit` kernel. Same `standalone_dual` key as the reliability runs.

In [1]:
import sys
from pathlib import Path

def _find_shared():
    hard = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/shared")
    for p in [Path.cwd(), Path.cwd().parent, hard, *Path.cwd().parents]:
        if (p / "config.py").exists() and p.name == "shared":
            return p
        cand = p / "editing" / "shared"
        if (cand / "config.py").exists():
            return cand
    return hard

SHARED = _find_shared()
EDITING = SHARED.parent
sys.path.insert(0, str(SHARED))
sys.path.insert(0, str(EDITING / "reliability"))

import config as C
sys.path.insert(0, str(C.DERM_EDIT_ROOT))

USERS = [
    "jesse.y.liu.md@gmail.com",
    "achiou@stanford.edu",
    "lhunterellul",
]
MODELS = ["medgemma", "dermatollama"]
RATIONALE_MODE = C.RATIONALE_MODE
NORM_ANSWER = "y16"
SUBSAMPLE = 0
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS
SEED = 42
OVERWRITE = False

EMBED_MODE = "standalone_dual"
IMAGE_ENCODER = "dinov2"
IMAGE_POOL = "cls"
TEXT_ENCODER = "sbert"
SCALER_DATASET = "coco"

OUT = EDITING / "text_generality" / "results"
print("shared:", SHARED)
print("results:", OUT)

shared: /scratch/jq2uw/MME/derm_vlm_edit/editing/shared
results: /scratch/jq2uw/MME/derm_vlm_edit/editing/text_generality/results


## Rephrased question sentences

Each sentence replaces only the opening request. The format suffix is the same string on every prompt.

In [2]:
FORMAT = (
    "in format:\n"
    "1. [Diagnosis]: [Reasoning]\n"
    "2. [Diagnosis]: [Reasoning]\n"
    "3. [Diagnosis]: [Reasoning]"
)

SENTENCES = [
    "As a board-certified dermatologist, what are the three most likely diagnoses for this skin finding",
    "Acting as a dermatology specialist, what are your top three differential diagnoses",
    "You are a skilled dermatologist examining this image. What three diagnoses lead your differential",
    "From the perspective of an experienced dermatologist, which three conditions are most likely",
    "Imagine you are a consulting dermatologist. What are your three leading diagnostic possibilities",
    "Given your expertise in dermatology, which three diagnoses would you rank as most likely",
    "As an attending dermatologist, what three diagnoses would you consider first",
    "You are a dermatologist evaluating this clinical image. What is your top-three differential",
    "Please assume the role of an expert in dermatology. Which three diagnoses are most plausible",
    "In your capacity as a dermatology expert, what three diagnoses would you consider most likely",
]

def make_question(sentence):
    return f"{sentence.strip()}, {FORMAT}"

print(f"{len(SENTENCES)} sentences")
print("---")
print(make_question(SENTENCES[-1]))

10 sentences
---
In your capacity as a dermatology expert, what three diagnoses would you consider most likely, in format:
1. [Diagnosis]: [Reasoning]
2. [Diagnosis]: [Reasoning]
3. [Diagnosis]: [Reasoning]


## Run

One model load per VLM. A finished JSON is reused unless `OVERWRITE` is set.

In [3]:
import json
import random

from IPython.display import Markdown, display
from data_engineering.build_edit_set import build as build_edit_set
from run_efficacy import is_correct

def _path(user, model_short):
    safe = user.replace("@", "_at_").replace(".", "_")
    return OUT / f"text_generality__{safe}__{model_short}.json"

def _reliability(user, model_short):
    safe = user.replace("@", "_at_").replace(".", "_")
    path = C.RESULTS_DIR / f"reliability_standalone__{safe}__{model_short}.json"
    if not path.exists():
        return None
    r = json.loads(path.read_text())
    n, k = int(r["n_edits"]), int(r["n_correct_post"])
    return k, n

def evaluate(user, model_short, vlm, config):
    from reasonedit import ReasonEdit
    from derm_dataset import DermEditDataset

    annot = C.MODEL_SHORT_TO_ANNOT[model_short]
    edits, _ = build_edit_set(user, annot, RATIONALE_MODE, write=False, norm_answer=NORM_ANSWER)
    if SUBSAMPLE:
        edits = edits[:SUBSAMPLE]
    if not edits:
        print(f"skip {user} / {model_short}")
        return None

    ds = DermEditDataset(config, edits)
    ds.set_dataloader()
    editor = ReasonEdit(config, vlm)
    editor.edit(config, edit_ds=ds)

    rng = random.Random(f"{SEED}|{user}|{model_short}")
    for ex in ds.data:
        sentence = rng.choice(SENTENCES)
        question = make_question(sentence)
        ex["rephrase"] = sentence
        ex["question"] = question
        ex["prompt"] = question
        ex["prompt_orig"] = question

    editor.apply_to_dataset(ds)
    ds.task_generate(vlm, use_cache=True, max_new_tokens=MAX_NEW_TOKENS)
    retr = {r["uid"]: r.get("facts", []) for r in (editor.last_retrieval_log or [])}

    rows, n_ok = [], 0
    for ex in ds.data:
        pred = ex.get("pred", {})
        ok = is_correct(pred.get("answer", ""), ex["answer"], ex.get("human_top1_y16"), NORM_ANSWER)
        n_ok += int(ok)
        rows.append({
            "uid": ex["uid"],
            "case_id": ex.get("case_id"),
            "rephrase": ex["rephrase"],
            "human_answer": ex["answer"],
            "human_y16": ex.get("human_top1_y16"),
            "pred": pred.get("answer", ""),
            "correct": ok,
            "retrieved": bool(retr.get(ex["uid"])),
        })

    n = len(rows)
    result = {
        "user": user,
        "model": annot,
        "short_model": model_short,
        "n": n,
        "n_correct": n_ok,
        "text_generality": n_ok / n if n else 0.0,
        "cases": rows,
    }
    OUT.mkdir(parents=True, exist_ok=True)
    _path(user, model_short).write_text(json.dumps(result, indent=2, ensure_ascii=False))
    print(f"{user} / {model_short}: {n_ok}/{n}")
    return result

pending = {}
for model_short in MODELS:
    todo = []
    for user in USERS:
        if _path(user, model_short).exists() and not OVERWRITE:
            print(f"{user} / {model_short}: loaded")
        else:
            todo.append(user)
    if todo:
        pending[model_short] = todo

if pending:
    import gc
    import torch
    from reasonedit import VQAModel, load_config
    from run_efficacy import resolve_layers

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    for model_short, todo in pending.items():
        config = load_config(
            model_short, device=device, batch_size=1,
            max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
            mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
            image_encoder_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
            scaler_dataset=SCALER_DATASET,
        )
        config.editor.fact_template = C.FACT_TEMPLATE
        vlm = VQAModel(config)
        resolve_layers(vlm, config)
        for user in todo:
            evaluate(user, model_short, vlm, config)
        del vlm
        gc.collect()
        torch.cuda.empty_cache()

results = []
for model_short in MODELS:
    for user in USERS:
        path = _path(user, model_short)
        if path.exists():
            results.append(json.loads(path.read_text()))

lines = ["| User | Model | Text generality | Reliability |",
         "| --- | --- | --- | --- |"]
for r in results:
    rel = _reliability(r["user"], r["short_model"])
    rel_cell = f"{rel[0]}/{rel[1]}" if rel else "--"
    lines.append(
        f"| {r['user'].split('@')[0]} | {r['model']} | {r['n_correct']}/{r['n']} | {rel_cell} |"
    )
display(Markdown("\n".join(lines)))

/scratch/jq2uw/conda_ex/conda_venv/dermedit/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|████████████████████████| 2/2 [00:02<00:00,  1.09s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[layers] inner_params_vision: ['model.vision_tower.vision_model.encoder.layers.24.mlp.fc1.weight']
[layers] inner_params_lang: ['model.language_model.layers.20.mlp.gate_proj.weight']
[layers] inner_params: ['model.language_model.layers.30.mlp.gate_proj.weight']
[ReasonEdit] balancing weight key: dinov2-large_cls__sbert__coco
[ReasonEdit] mode=standalone_dual, key_type=answer, pair_rationale_w=orig, merge=False, hubness=False, cap_keys=1
[ReasonEdit] edit 1/24...[DINOv2Encoder] Loading facebook/dinov2-large ...
[DINOv2Encoder] Loaded (1024-d, pool=cls)
[SentenceEncoder] Loading sentence-transformers/paraphrase-mpnet-base-v2 ...
[SentenceEncoder] Loaded sbert (768-d)
[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 2/24...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 3/24...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 4/24...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 5/24...[Keys] +1 added, 0 merged (from 1 candidat

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for o

jesse.y.liu.md@gmail.com / medgemma: 24/24
[ReasonEdit] balancing weight key: dinov2-large_cls__sbert__coco
[ReasonEdit] mode=standalone_dual, key_type=answer, pair_rationale_w=orig, merge=False, hubness=False, cap_keys=1
[ReasonEdit] edit 1/20...[DINOv2Encoder] Loading facebook/dinov2-large ...
[DINOv2Encoder] Loaded (1024-d, pool=cls)
[SentenceEncoder] Loading sentence-transformers/paraphrase-mpnet-base-v2 ...
[SentenceEncoder] Loaded sbert (768-d)
[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 2/20...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 3/20...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 4/20...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 5/20...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 6/20...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 7/20...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 8/20...[Keys] +1 added, 0 merged (from 1 candidates)
[R

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for o

...

Loading checkpoint shards: 100%|████████████████████████| 5/5 [00:12<00:00,  2.54s/it]


  Suffix-matched layer: base_model.model.model.vision_model.transformer.layers.28.mlp.fc1.weight
[layers] inner_params_vision: ['base_model.model.model.vision_model.transformer.layers.28.mlp.fc1.weight']
  Found corrected layer: base_model.model.model.language_model.layers.0.mlp.gate_proj.base_layer.weight
[layers] inner_params_lang: ['base_model.model.model.language_model.layers.0.mlp.gate_proj.base_layer.weight']
  Found corrected layer: base_model.model.model.language_model.layers.0.mlp.gate_proj.base_layer.weight
[layers] inner_params: ['base_model.model.model.language_model.layers.0.mlp.gate_proj.base_layer.weight']
[ReasonEdit] balancing weight key: dinov2-large_cls__sbert__coco
[ReasonEdit] mode=standalone_dual, key_type=answer, pair_rationale_w=orig, merge=False, hubness=False, cap_keys=1
[ReasonEdit] edit 1/14...[DINOv2Encoder] Loading facebook/dinov2-large ...
[DINOv2Encoder] Loaded (1024-d, pool=cls)
[SentenceEncoder] Loading sentence-transformers/paraphrase-mpnet-base-v2 ..

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


jesse.y.liu.md@gmail.com / dermatollama: 14/14
[ReasonEdit] balancing weight key: dinov2-large_cls__sbert__coco
[ReasonEdit] mode=standalone_dual, key_type=answer, pair_rationale_w=orig, merge=False, hubness=False, cap_keys=1
[ReasonEdit] edit 1/13...[DINOv2Encoder] Loading facebook/dinov2-large ...
[DINOv2Encoder] Loaded (1024-d, pool=cls)
[SentenceEncoder] Loading sentence-transformers/paraphrase-mpnet-base-v2 ...
[SentenceEncoder] Loaded sbert (768-d)
[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 2/13...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 3/13...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 4/13...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 5/13...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 6/13...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 7/13...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 8/13...[Keys] +1 added, 0 merged (from 1 candidates

| User | Model | Text generality | Reliability |
| --- | --- | --- | --- |
| jesse.y.liu.md | MedGemma | 24/24 | 24/24 |
| achiou | MedGemma | 19/20 | 20/20 |
| lhunterellul | MedGemma | 8/8 | 8/8 |
| jesse.y.liu.md | DermatoLlama | 14/14 | 14/14 |
| achiou | DermatoLlama | 13/13 | 13/13 |
| lhunterellul | DermatoLlama | 6/6 | 6/6 |